# Government Bond Yields: US, Japan, Germany, UK, Australia and China

Daily 10-year and 30-year benchmark government bond yields for five major
markets, a separate 10-year chart putting China alongside the US, and the
Australian 10-year yield less the US 10-year yield as a spread in its own
right.

A final section goes behind the quoted yields, to the AOFM's and the New York
Fed's decompositions of the Australian and US curves into an expected short-rate
path and a term premium, and charts the 5y5y risk-neutral forward that falls out
of them. It has its own sources and caveats, set out where it begins.

**Sources**

| Country | Source | 10-year from | 30-year from |
|---|---|---|---|
| United States | Yahoo Finance `^TNX` / `^TYX` constant-maturity indices | 1962 | 1977 |
| Japan | Ministry of Finance daily JGB curve (CSV) | Jul 1986 | Sep 1999 |
| Germany | Bundesbank daily term structure of listed Federal securities | Aug 1997 | Aug 2000 |
| United Kingdom | Bank of England fitted nominal gilt curve | 1979 | **2016** |
| Australia | RBA table F2 (daily), spliced to the pre-2013 historical table | 1995 | **none published** |
| China | ChinaBond daily sovereign (CGB) yield curve | Mar 2006 | not charted |

Each line begins when its own data begins; no series is truncated to a common
window. The charts themselves start in 1986 (10-year), 1999 (30-year) and 2006
(China against the US).

**Caveats**

- **China is charted separately, against the US alone.** Its history starts in
  2006, decades after the others, and a sixth line makes the five-country chart
  unreadable. ChinaBond publishes a 30-year point too, but China's 30-year
  market is thin and the comparison would be a weaker one, so only the 10-year
  is plotted.
- **Australia appears at ten years only.** The RBA's capital market yields stop
  at a 10-year maturity, and the AOFM publishes no daily secondary-market yield
  curve, so there is no published Australian 30-year constant-maturity yield to
  plot. Australia issued its first 30-year bond only in 2016.
- **The AU less US spread is taken only on days both markets traded.** The two
  keep different holidays and publish with different lags, so differencing the
  raw union index would net an Australian yield against a stale American one.
  The unmatched days are dropped, and the spread therefore ends on the last day
  both series exist, which is the earlier of the two end dates.
- **These are different constructs.** US yields are constant-maturity
  (par-equivalent), Japan's are yields on benchmark bonds, Germany's, the UK's
  and China's are fitted curves, and Australia's are the RBA's interpolated
  bond yields. At ten years the differences are minor; at thirty years spot and
  par yields can diverge noticeably. The comparison is the right one to make,
  but the series are not computed identically.
- **UK 30-year only reaches back to 2016.** The BoE's pre-2016 workbooks stop at
  a 25-year maturity, so there is no 30-year gilt yield to plot before then.
- **US 30-year, Feb 2002 to Feb 2006.** Treasury issued no 30-year bond over that
  stretch. FRED's `DGS30` has a genuine four-year hole; `^TYX` does not, because
  CBOE kept quoting off the longest bond still outstanding, whose maturity was
  drifting down toward 25 years. The 30-year chart carries this as a header note.
- Publication lags differ by a few days, so the lines need not end on the same
  date.

**Downloading and caching**

Everything is cached in the project's shared `./CACHE`, under the usual
`prefix--hash--filename` convention.

- The **MOF** files go through `readabs.download_cache.get_file()`: they send
  `Last-Modified`, so it re-downloads only on a new publication and falls back
  to the cached copy if a download fails. The **AOFM** and **NY Fed** workbooks
  in the final section use it for the same reason.
- The **Bundesbank**, **Bank of England** and **ChinaBond** cannot use
  `get_file()`, for two different reasons, so they share `fetch_cached()` in the
  next cell. The BoE answers a request without a User-Agent with a 403, and
  `get_file()` cannot be given headers. The Bundesbank sends no `Last-Modified`
  at all, so `get_file()` would serve its first download forever and the German
  line would silently stop updating. `fetch_cached()` downloads fresh, caches,
  and falls back to the cached copy when a download fails - which matters
  because the Bundesbank throttles bursts of requests behind a proof-of-work
  challenge, and without the fallback a throttle takes the whole notebook down.
- The BoE history archive is about 39 MB and only changes as years roll over, so
  it alone is fetched conditionally on the server's `Last-Modified`.
- **ChinaBond** caps a query at a one-year window, so its history is assembled a
  calendar year at a time - about twenty requests. It sends no useful
  `Last-Modified` either, but a completed year cannot change, so past years are
  read straight from the cache and only the current year is re-downloaded.

**Other source quirks:** the MOF files served from the `/english/` path still
carry Shift-JIS bytes, so they are decoded as `cp932`; the Bundesbank CSVs put a
variable number of metadata rows ahead of the data; the BoE's current workbook
carries a literal "Refresh" row, which leaves the yield columns as object dtype
unless they are coerced - an object column plots but silently loses its
end-point label; and ChinaBond answers with a rendered HTML table carrying three
curves at once, so rows are filtered to the sovereign curve by name.

## Setup

In [1]:
# system imports
import io
import re
import zipfile
from datetime import UTC, datetime
from functools import cache
from hashlib import sha256
from pathlib import Path
from typing import cast

# analytic imports
import pandas as pd
import readabs as ra
import requests
import yfinance as yf
from readabs import metacol as mc
from readabs.download_cache import (
    BAD_CACHE_PATTERN,
    get_file,
    retrieve_from_cache,
    save_to_cache,
)

# local imports
import mgplot as mg

In [2]:
# pandas display
pd.options.display.max_rows = 999999

# chart output directory
CHART_DIR = "./CHARTS/Bonds/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

# display charts inline?
SHOW = False

# Chart windows: full history, and roughly the last three years of trading days.
RECENT_TRADING_DAYS = -750
plot_times = 0, RECENT_TRADING_DAYS

# Series code per tenor and country: a Yahoo ticker (US), a MOF curve column
# (Japan), a Bundesbank series key (Germany), a BoE curve maturity in years
# (UK), or an RBA series title (Australia). Countries start when their data
# starts; no series is truncated to a common window. Australia appears only at
# ten years - neither the RBA nor the AOFM publishes a 30-year yield.
CODES: dict[str, dict[str, str]] = {
    "10-year": {
        "United States": "^TNX",
        "Japan": "10Y",
        "Germany": "D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A",
        "United Kingdom": "10",
        "Australia": "Australian Government 10 year bond",
    },
    "30-year": {
        "United States": "^TYX",
        "Japan": "30Y",
        "Germany": "D.I.ZST.ZI.EUR.S1311.B.A604.R30XX.R.A.A._Z._Z.A",
        "United Kingdom": "30",
    },
}

# Shorter names for chart titles; anything absent is used as it stands.
SHORT_NAMES: dict[str, str] = {
    "United States": "US",
    "United Kingdom": "UK",
}

# Two-letter codes for the footer, which names every series at once.
ABBREVIATIONS: dict[str, str] = {
    "United States": "US",
    "Japan": "JP",
    "Germany": "DE",
    "United Kingdom": "UK",
    "Australia": "AU",
    "China": "CN",
}

# Who publishes each country's yield, for the chart's source footer. Named per
# country rather than once for the notebook, because not every chart carries
# every country.
SOURCES: dict[str, str] = {
    "United States": "Yahoo Finance",
    "Japan": "Japan MOF",
    "Germany": "Bundesbank",
    "United Kingdom": "Bank of England",
    "Australia": "RBA",
    "China": "ChinaBond",
}

# Where each chart begins. The 10-year window opens when Japan joins in 1986;
# the 30-year when the JGB 30-year sector opened in 1999.
TENOR_STARTS: dict[str, str] = {
    "10-year": "1986-01-01",
    "30-year": "1999-01-01",
}

# Comparability caveats, shown as that chart's rheader.
TENOR_NOTES: dict[str, str | None] = {
    "10-year": None,
    "30-year": (
        "US Feb 2002 to Feb 2006: no 30-year issuance, so ^TYX tracks the "
        "longest bond outstanding. UK curve reaches 30 years only from 2016"
    ),
}

# China against the US at ten years, as its own chart. China is left out of the
# five-country charts because its history starts in 2006, decades after theirs,
# and because a sixth line makes that chart unreadable.
CHINA_US_CODES: dict[str, str] = {
    "China": "10",
    "United States": "^TNX",
}
CHINA_US_TENOR = "10-year"
CHINA_US_START = "2006-01-01"
CHINA_US_NOTE = (
    "China: ChinaBond fitted CGB curve. US: ^TNX constant maturity"
)

# The Australia less US 10-year spread, taken from the two columns the 10-year
# frame already carries. It begins where the shorter of the two series does -
# the RBA's daily yields, in 1995.
SPREAD_TENOR = "10-year"
SPREAD_LEGS = ("Australia", "United States")
SPREAD_NOTE = (
    "RBA interpolated 10-year yield less the ^TNX constant maturity yield: "
    "the same tenor, but not identically computed"
)

# The earliest date asked of Yahoo Finance; each chart's own start does the
# trimming.
EARLIEST_START = "1960-01-01"

# The project's shared download cache, and a prefix per source.
CACHE_DIR = Path("./CACHE")
MOF_CACHE_PREFIX = "jgb_curve"
BBK_CACHE_PREFIX = "bund_yields"
BOE_CACHE_PREFIX = "boe_yield_curve"
CBD_CACHE_PREFIX = "chinabond_curve"

# The Bank of England answers a header-less request with a 403.
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}
HEAD_TIMEOUT = 30
DOWNLOAD_TIMEOUT = 300

# Japan MOF daily JGB yield curve: a full history file plus the current month.
MOF_BASE = "https://www.mof.go.jp/english/policy/jgbs/reference/interest_rate/"
MOF_HISTORY = "historical/jgbcme_all.csv"
MOF_CURRENT = "jgbcme.csv"
MOF_ENCODING = "cp932"  # the "English" CSVs still contain Shift-JIS bytes

# Bundesbank daily term structure of listed Federal securities (Svensson).
BBK_BASE = "https://api.statistiken.bundesbank.de/rest/download/BBSIS/"

# Bank of England fitted nominal gilt curve. The history archive is ~39 MB, so
# it is only re-downloaded when the server reports a newer copy; the
# current-month archive is small and fetched every run.
BOE_BASE = "https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/"
BOE_HISTORY = "glcnominalddata.zip"
BOE_CURRENT = "latest-yield-curve-data.zip"

# ChinaBond's daily yield curves, queried a calendar year at a time - the
# service answers a longer window with an empty page rather than an error. The
# curve ID selects the sovereign (CGB) curve; the reply carries the bank and
# mid-term note curves alongside it, so rows are also filtered on the curve
# name. The 10-year point begins on 1 March 2006.
CBD_URL = "https://yield.chinabond.com.cn/cbweb-cbrc-web/cbrc/historyQuery"
CBD_CURVE_ID = "2c9081e50a2f9606010a3068cae70001"
CBD_CURVE_NAME = "中债国债收益率曲线"
CBD_NAME_COLUMN = "曲线名称"
CBD_DATE_COLUMN = "日期"
CBD_FIRST_YEAR = 2006

# RBA daily capital market yields, split into a current and a pre-2013 table.
RBA_CURRENT_TABLE = "F2"
RBA_HISTORY_TABLE = "Z:F2-Daily-2013"

# Metadata rows precede the data in the MOF and Bundesbank files, and the count
# is not constant, so data rows are recognised by their leading date.
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2},")

# --- Risk-neutral forwards ---------------------------------------------------

# AOFM's daily decomposition of the nominal Treasury Bond curve into a fitted
# zero-coupon yield (FY), a term premium (TP) and a risk-neutral yield (RNY), at
# every tenor from 1 to 10 years, from July 1992. The datestamp in the path is
# the node's creation date, not the data vintage: the file is updated in place,
# so the copy behind this URL carries data well past 2025-06-06.
AOFM_URL = "https://www.aofm.gov.au/sites/default/files/2025-06-06/term%20premium.xlsx"
AOFM_HUB = "https://www.aofm.gov.au/data-hub"
AOFM_CACHE_PREFIX = "aofm_term_premium"
# "bc" bias-corrects the VAR parameters, "ols" is plain Adrian-Crump-Moench.
# ACM's three-step regression inherits the small-sample downward bias in the
# persistence of a highly autocorrelated VAR, which pushes variation out of the
# expectations component and into the premium, so the bias-corrected sheet is
# the default here as it is at the AOFM.
AOFM_METHOD_SHEETS: dict[str, str] = {"bc": "TermPremiumBC", "ols": "TermPremiumOLS"}
AOFM_METHOD = "bc"

# The New York Fed's ACM decomposition of the US Treasury curve: the same object
# as the AOFM file, so the two countries' forwards are comparable in a way two
# 10-year yields are not. The workbook holds an "ACM Monthly" sheet and an
# "ACM Daily" sheet, both back to 1961, and the daily one is read: it lets the
# US leg be averaged over the month exactly as the Australian leg is, rather
# than setting a month's mean against a single end-month observation. Naming
# the sheet also matters because "ACM Monthly" comes first in the workbook, so
# the default is the monthly one by position rather than by choice.
# Plain ACM - the Fed publishes no bias-corrected variant.
ACM_URL = (
    "https://www.newyorkfed.org/medialibrary/media/research/"
    "data_indicators/ACMTermPremium.xls"
)
ACM_DAILY_SHEET = "ACM Daily"
ACM_CACHE_PREFIX = "nyfed_acm"

# The RBA's announced Cash Rate Target: the policy rate itself, a step function
# that moves only when the Board decides. `readabs.read_rba_ocr` reads it from
# table A2 (series ARBAMPCNCRT) and keeps the last announced value in each month.
# NOT F1.1's "Interbank Overnight Cash Rate", which is a monthly MEAN of the
# realised overnight rate and so slopes through any month a decision lands in:
# over 1993-2026 the two differ by more than 0.05pp in 66 of 404 months, by as
# much as 0.79pp. The target starts in August 1990, well before this chart does.
RBA_CASH_TABLE = "A2"

# Real GDP per capita, for the golden-rule reference line. The trailing " ;"
# matters: without it the description also matches the published
# "- Percentage changes" item, and find_abs_id would be ambiguous.
ABS_NA_CAT = "5206.0"
ABS_NA_TABLE = "5206001_Key_Aggregates"
GDP_PER_CAPITA_DID = "GDP per capita: Chain volume measures ;"

# The midpoint of the RBA's 2-3 per cent target band. Flat, not survey
# expectations: the golden-rule statement is real growth plus the inflation the
# central bank is aiming at, so the target IS the second term, and substituting
# what people expect would make the benchmark drift with sentiment.
INFLATION_TARGET = 2.5
# Quarters in the rolling window behind trend per-capita growth. Ten years, so
# the window spans a cycle and neither the mining boom nor the pandemic owns it.
TREND_WINDOW_QTRS = 40
# A period counts as complete if it holds at least this share of the median
# number of trading days. Both forward series run to the current month, so the
# last month is otherwise a mean over however many trading days have happened.
COMPLETE_PERIOD_SHARE = 0.8

# Where each of the two forward charts begins.
RSTAR_START = "1993-01"
CORRELATION_START = "1992-07"

# A note on the golden-rule line, boxed in the empty bottom-left corner of the
# r* chart. The inflation term in that line is the target, held flat from 1993,
# but expectations only settled onto the target through the second half of the
# 1990s - so the line understates nominal neutral over its first years, and a
# reader who does not know that reads the early gap as the market disagreeing
# with fundamentals. Placed in axes fractions, not data coordinates, so it
# stays in the corner whatever the axis limits do. "About 1998" is deliberately
# vague: the convergence was gradual, and a quarter would imply a precision
# nothing on the chart supports.
RSTAR_NOTE = (
    "The orange line would have been pinned by\n"
    "inflation expectations, which did not settle at\n"
    "the RBA's target until around 1998 - so before\n"
    "then it sits too low."
)
RSTAR_NOTE_XY = (0.02, 0.04)
RSTAR_NOTE_FONTSIZE = "small"

# One label per series, because the Australian forward appears on both charts
# and a reader moving between them must be able to see it is the same series.
FORWARD_LABEL = "AOFM 5y5y risk-neutral forward"
US_FORWARD_LABEL = "US ACM 5y5y risk-neutral forward"
TREND_G_LABEL = (
    f"Trend GDP per capita growth ({TREND_WINDOW_QTRS // 4}yr avg) "
    f"+ {INFLATION_TARGET}% target"
)

## Data capture

In [3]:
def cache_path(url: str, prefix: str) -> Path:
    """The readabs cache file name for a URL, so these downloads sit in ./CACHE
    under the same prefix--hash--filename convention as everything else."""
    digest = sha256(url.encode("utf-8")).hexdigest()
    tail = url.rsplit("/", 1)[-1].split("?", 1)[0]
    return CACHE_DIR / re.sub(BAD_CACHE_PATTERN, "", f"{prefix}--{digest}--{tail}")


def fetch_cached(url: str, prefix: str, *, conditional: bool = False) -> bytes:
    """Download a URL that readabs' own fetchers cannot handle, and cache it.

    `get_file()` is the right tool wherever it works, and the MOF files use it.
    It is wrong for the other two sources here:

    - The Bank of England answers a request without a User-Agent with a 403,
      and neither `get_file()` nor `request_get()` can be given headers.
    - The Bundesbank sends no Last-Modified header. With nothing to compare
      against, `get_file()` returns its first download forever, so the German
      series would silently stop updating.

    So this downloads afresh, caches the bytes, and falls back to the cached
    copy when the download fails - which is what stops a transient block (the
    Bundesbank throttles bursts of requests) from failing the whole notebook.
    `conditional` skips the download when the server reports nothing newer than
    the cached copy, which is what spares re-fetching the 39 MB BoE archive.
    """
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = cache_path(url, prefix)

    if conditional and path.exists():
        head = requests.head(
            url, headers=BROWSER_HEADERS, allow_redirects=True, timeout=HEAD_TIMEOUT
        )
        modified = head.headers.get("Last-Modified")
        cached_at = pd.Timestamp(
            datetime.fromtimestamp(path.stat().st_mtime, tz=UTC)
        )
        if modified is not None and pd.to_datetime(modified, utc=True) <= cached_at:
            return retrieve_from_cache(path)

    try:
        response = requests.get(
            url, headers=BROWSER_HEADERS, timeout=DOWNLOAD_TIMEOUT
        )
        response.raise_for_status()
    except requests.RequestException as error:
        if not path.exists():
            raise
        print(f"Download failed ({error}); using the cached copy of {url}")
        return retrieve_from_cache(path)

    save_to_cache(path, response.content)
    return response.content

In [4]:
def fetch_us_yield(code: str) -> pd.Series:
    """Fetch a daily US Treasury constant-maturity yield from Yahoo Finance."""
    raw = yf.download(code, start=EARLIEST_START, auto_adjust=True, progress=False)
    if raw is None or len(raw) == 0:
        raise ValueError(f"Yahoo Finance returned no data for {code}")
    series = raw["Close"].squeeze().dropna()
    series.index = cast("pd.DatetimeIndex", series.index).to_period("D")
    return series


def fetch_bund_yield(code: str) -> pd.Series:
    """Fetch a daily Bund yield from the Bundesbank time-series API.

    The CSV carries a variable number of metadata rows ahead of the data, and
    publishes non-trading days as a bare full stop.
    """
    url = f"{BBK_BASE}{code}?format=csv&lang=en"
    content = fetch_cached(url, BBK_CACHE_PREFIX)
    lines = content.decode("utf-8-sig").splitlines()
    observations = {
        pd.Period(row[0], freq="D"): float(row[1])
        for row in (line.split(",") for line in lines if ISO_DATE.match(line))
        if row[1] not in (".", "")
    }
    if not observations:
        raise ValueError(f"Bundesbank returned no observations for {code}")
    return pd.Series(observations).sort_index()

In [5]:
def _fetch_mof_csv(path: str) -> pd.DataFrame:
    """Read one MOF JGB yield-curve CSV into a dated frame of per-cent yields.

    These files send Last-Modified and need no headers, so readabs' get_file()
    handles them: fresh when the MOF has published, cached otherwise, and the
    cached copy is used if a download fails. Row 0 is a title line, so the
    header is on row 1, and unquoted tenors are published as a bare hyphen.
    """
    content = get_file(
        MOF_BASE + path, cache_dir=CACHE_DIR, cache_prefix=MOF_CACHE_PREFIX
    )
    text = content.decode(MOF_ENCODING)
    frame = pd.read_csv(io.StringIO(text), skiprows=1, na_values=["-"])
    frame["Date"] = pd.to_datetime(frame["Date"], format="%Y/%m/%d", errors="coerce")
    frame = frame.dropna(subset=["Date"]).set_index("Date")
    return frame.apply(pd.to_numeric, errors="coerce")


@cache
def mof_curve() -> pd.DataFrame:
    """The whole daily JGB curve: the MOF history file, extended by the current
    month. Cached, so every tenor is served from one pair of downloads."""
    history = _fetch_mof_csv(MOF_HISTORY)
    current = _fetch_mof_csv(MOF_CURRENT)
    curve = pd.concat(
        [history, current[~current.index.isin(history.index)]]
    ).sort_index()
    if curve.empty:
        raise ValueError("MOF returned an empty yield curve")
    curve.index = pd.PeriodIndex(curve.index, freq="D")
    return curve


def fetch_japan_yield(code: str) -> pd.Series:
    """Fetch one tenor of the daily JGB yield curve."""
    curve = mof_curve()
    if code not in curve.columns:
        raise ValueError(f"MOF curve has no {code} column: {list(curve.columns)}")
    series = curve[code].dropna()
    if series.empty:
        raise ValueError(f"MOF returned no {code} observations")
    return series

In [6]:
def _boe_spot_curve(archive_bytes: bytes) -> pd.DataFrame:
    """Read the nominal spot curve out of every workbook in a BoE archive.

    The archives hold real and inflation curves too, and the spot sheet was
    renamed from "4. nominal spot curve" to "4. spot curve" from the 2005
    workbook onward, so the sheet is matched on its suffix. Row 3 of the sheet
    holds the maturities in years.

    The current workbook carries a literal "Refresh" row where a date should
    be. Coercing the dates drops that row, but the yield columns stay object
    dtype unless they are coerced too - and an object column plots as a line
    while silently losing its end-point annotation.
    """
    archive = zipfile.ZipFile(io.BytesIO(archive_bytes))
    frames = []
    for name in archive.namelist():
        if "Nominal" not in name:
            continue
        book = pd.ExcelFile(io.BytesIO(archive.read(name)))
        sheets = [
            sheet
            for sheet in book.sheet_names
            if sheet.endswith("spot curve") and "short end" not in sheet
        ]
        if not sheets:
            raise ValueError(f"No nominal spot curve sheet in {name}")
        frame = book.parse(sheets[0], header=3)
        frame = frame.rename(columns={frame.columns[0]: "Date"})
        frame["Date"] = pd.to_datetime(
            frame["Date"], errors="coerce", format="mixed"
        )
        frame = frame.dropna(subset=["Date"]).set_index("Date")
        frames.append(frame.apply(pd.to_numeric, errors="coerce"))
    if not frames:
        raise ValueError("BoE archive held no nominal curve workbook")
    return pd.concat(frames).sort_index()


@cache
def boe_curve() -> pd.DataFrame:
    """The daily fitted nominal gilt curve: the history archive plus this month.

    The history archive is ~39 MB and changes only as years roll over, so it is
    fetched conditionally on the server's Last-Modified.
    """
    history = _boe_spot_curve(
        fetch_cached(BOE_BASE + BOE_HISTORY, BOE_CACHE_PREFIX, conditional=True)
    )
    current = _boe_spot_curve(
        fetch_cached(BOE_BASE + BOE_CURRENT, BOE_CACHE_PREFIX)
    )
    curve = pd.concat(
        [history, current[~current.index.isin(history.index)]]
    ).sort_index()
    curve.index = pd.PeriodIndex(curve.index, freq="D")
    return curve


def fetch_gilt_yield(code: str) -> pd.Series:
    """Fetch one maturity off the BoE nominal gilt curve, in years."""
    curve = boe_curve()
    maturity = float(code)
    if maturity not in curve.columns:
        raise ValueError(f"BoE curve has no {maturity}-year maturity")
    series = curve[maturity].dropna()
    if series.empty:
        raise ValueError(f"BoE returned no {maturity}-year observations")
    return series

In [7]:
def fetch_australia_yield(code: str) -> pd.Series:
    """Fetch a daily RBA Australian Government bond yield, named by its title.

    The RBA split its daily capital market yields in 2013, so the current table
    is extended backwards by the historical one. The series ID is resolved from
    the current table's own metadata rather than written down here, since the
    two tables share IDs but not metadata layouts.
    """
    current, meta = ra.read_rba_table(RBA_CURRENT_TABLE)
    matches = meta[meta["Title"] == code]
    if len(matches) != 1:
        raise ValueError(
            f"RBA {RBA_CURRENT_TABLE} holds {len(matches)} series titled {code!r}"
        )
    series_id = str(matches["Series ID"].iloc[0])
    history, _ = ra.read_rba_table(RBA_HISTORY_TABLE)
    if series_id not in history.columns:
        raise ValueError(f"RBA {RBA_HISTORY_TABLE} has no {series_id} column")
    # keep="last" lets the current table win wherever the two tables overlap
    joined = pd.concat([history[series_id].dropna(), current[series_id].dropna()])
    joined = joined[~joined.index.duplicated(keep="last")].sort_index()
    if joined.empty:
        raise ValueError(f"RBA returned no observations for {code!r}")
    return joined

In [8]:
def _chinabond_year(year: int, code: str) -> pd.Series:
    """One calendar year of the ChinaBond sovereign curve at one maturity.

    The service caps a query at a one-year window and answers a longer one with
    an empty page rather than an error, so the history is assembled a year at a
    time. It sends no useful Last-Modified, so a completed year - which cannot
    change - is read straight from the cache and never re-downloaded; only the
    current year goes back to the server.

    The reply is a server-rendered page whose data table carries its header in
    the first row, and holds the bank and mid-term note curves alongside the
    sovereign one, so rows are filtered on the curve name.
    """
    url = (
        f"{CBD_URL}?startDate={year}-01-01&endDate={year}-12-31"
        f"&gjqx={code}&qxId=ycqx&locale=zh_CN&wrjxCBFlag=0"
        f"&ycDefIds={CBD_CURVE_ID}"
    )
    path = cache_path(url, CBD_CACHE_PREFIX)
    complete = year < datetime.now(tz=UTC).year
    content = (
        retrieve_from_cache(path)
        if complete and path.exists()
        else fetch_cached(url, CBD_CACHE_PREFIX)
    )

    tenor = f"{code}年"
    frames = []
    for table in pd.read_html(io.StringIO(content.decode("utf-8"))):
        frame = table.set_axis([str(name) for name in table.iloc[0]], axis=1)[1:]
        if CBD_NAME_COLUMN not in frame.columns or tenor not in frame.columns:
            continue
        frames.append(frame[frame[CBD_NAME_COLUMN] == CBD_CURVE_NAME])
    if not frames:
        if complete:
            raise ValueError(f"ChinaBond returned no {year} data table")
        return pd.Series(dtype=float, index=pd.PeriodIndex([], freq="D"))

    rows = pd.concat(frames)
    return pd.Series(
        pd.to_numeric(rows[tenor], errors="coerce").to_numpy(),
        index=pd.PeriodIndex(rows[CBD_DATE_COLUMN], freq="D"),
    ).dropna()


def fetch_china_yield(code: str) -> pd.Series:
    """Fetch the daily ChinaBond sovereign (CGB) yield at one maturity."""
    years = range(CBD_FIRST_YEAR, datetime.now(tz=UTC).year + 1)
    series = pd.concat([_chinabond_year(year, code) for year in years]).sort_index()
    series = series[~series.index.duplicated(keep="last")]
    if series.empty:
        raise ValueError(f"ChinaBond returned no {code}-year observations")
    return series

In [9]:
def fetch_yield(country: str, code: str) -> pd.Series:
    """Fetch one country's yield series from whichever source publishes it.

    The result is forced to a numeric dtype here rather than in each fetcher.
    Several sources hand back object columns - the RBA tables do, and so does a
    BoE workbook carrying its "Refresh" row - and an object column plots as a
    line while mgplot silently skips its end-point annotation. Coercing in one
    place means a new source cannot reintroduce that quietly.
    """
    fetchers = {
        "United States": fetch_us_yield,
        "Japan": fetch_japan_yield,
        "Germany": fetch_bund_yield,
        "United Kingdom": fetch_gilt_yield,
        "Australia": fetch_australia_yield,
        "China": fetch_china_yield,
    }
    if country not in fetchers:
        raise ValueError(f"No fetcher defined for {country}")
    series = pd.to_numeric(fetchers[country](code), errors="coerce").dropna()
    if series.empty:
        raise ValueError(f"No numeric observations for {country} ({code})")
    return series.rename(country)


def get_yields(name: str, codes: dict[str, str], start: str) -> pd.DataFrame:
    """Assemble one tenor's country series onto a common daily index.

    The markets keep different holidays and their histories begin at different
    dates, so the union index carries gaps in every column; they are left as
    missing rather than filled.
    """
    frame = pd.DataFrame(
        {country: fetch_yield(country, code) for country, code in codes.items()}
    )
    frame = frame[frame.index >= pd.Period(start, freq="D")].dropna(how="all")
    if frame.empty:
        raise ValueError(f"No {name} observations on or after {start}")
    for country in frame.columns:
        column = frame[country].dropna()
        print(
            f"{name} {country}: {column.index[0]} to {column.index[-1]}, "
            f"{len(column)} observations, last {column.iloc[-1]:.2f} per cent"
        )
    return frame


def get_all_yields() -> dict[str, pd.DataFrame]:
    """Fetch every specified tenor."""
    return {
        name: get_yields(name, codes, TENOR_STARTS[name])
        for name, codes in CODES.items()
    }


yields = get_all_yields()
china_us = get_yields(CHINA_US_TENOR, CHINA_US_CODES, CHINA_US_START)

10-year United States: 1986-01-02 to 2026-09-21, 10216 observations, last 4.96 per cent
10-year Japan: 1986-07-05 to 2026-09-17, 9942 observations, last 2.99 per cent
10-year Germany: 1997-08-07 to 2026-09-21, 7392 observations, last 3.52 per cent
10-year United Kingdom: 1986-01-02 to 2026-09-18, 10290 observations, last 5.29 per cent
10-year Australia: 1995-01-03 to 2026-09-16, 7995 observations, last 5.35 per cent


30-year United States: 1999-01-04 to 2026-09-21, 6963 observations, last 5.30 per cent
30-year Japan: 1999-09-02 to 2026-09-17, 6626 observations, last 4.05 per cent
30-year Germany: 2000-08-01 to 2026-09-21, 6640 observations, last 3.88 per cent
30-year United Kingdom: 2016-01-04 to 2026-09-18, 2707 observations, last 5.75 per cent


10-year China: 2006-03-01 to 2026-09-21, 5144 observations, last 1.68 per cent
10-year United States: 2006-01-03 to 2026-09-21, 5208 observations, last 4.96 per cent


## Plotting

In [10]:
def join_names(names: list[str]) -> str:
    """Join names as a readable list: "a, b and c"."""
    if len(names) == 1:
        return names[0]
    return f"{', '.join(names[:-1])} and {names[-1]}"


def title_countries(countries: list[str]) -> str:
    """Render a country list for a chart title, shortening the long names."""
    return join_names([SHORT_NAMES.get(country, country) for country in countries])


def data_to_footer(data: pd.DataFrame) -> str:
    """Footer text giving the last observation date of every series.

    These markets publish with different lags, so one "data to" date would
    overstate the freshness of whichever series ended earliest. Countries are
    two-letter coded and those sharing an end date are grouped, most recent
    first.
    """
    ends: dict[pd.Period, list[str]] = {}
    for country in data.columns:
        code = ABBREVIATIONS.get(country, country)
        ends.setdefault(data[country].dropna().index[-1], []).append(code)
    parts = [
        f"{join_names(codes)} {end.strftime('%-d-%b-%Y')}"
        for end, codes in sorted(ends.items(), reverse=True)
    ]
    return f"Data to: {'; '.join(parts)}. "


def source_footer(data: pd.DataFrame) -> str:
    """Footer text naming the publisher of every series on this chart."""
    return "Sources: " + "; ".join(SOURCES[country] for country in data.columns)


def plot_yields(name: str, data: pd.DataFrame, note: str | None) -> None:
    """Chart one tenor over the full history and the recent window.

    mgplot type-checks `rheader` as a string, so a tenor with no comparability
    caveat omits the argument rather than passing None.
    """
    header = {"rheader": note} if note is not None else {}
    mg.multi_start(
        data,
        function=mg.line_plot_finalise,
        starts=plot_times,
        title=f"{title_countries(list(data.columns))}: {name} Government Bond Yields",
        ylabel="Per cent per year",
        xlabel=None,
        width=1,
        annotate=True,
        rounding=2,
        legend={"loc": "best", "fontsize": "small"},
        lfooter=data_to_footer(data),
        rfooter=source_footer(data),
        show=SHOW,
        **header,
    )


def plot_spread(
    name: str, data: pd.DataFrame, legs: tuple[str, str], note: str
) -> None:
    """Chart the yield spread between two of a tenor's countries.

    The two markets keep different holidays, so the difference is taken only on
    days both of them traded: dropping the unmatched days leaves an honest gap
    rather than a spread against a stale yield. The footers describe that pair,
    so the "data to" date is the last day the spread itself exists.
    """
    first, second = legs
    pair = data[[first, second]].dropna()
    if pair.empty:
        raise ValueError(f"No {name} days on which both {first} and {second} traded")
    spread = pair[first] - pair[second]
    mg.multi_start(
        spread,
        function=mg.line_plot_finalise,
        starts=plot_times,
        title=(
            f"{title_countries([first])} less {title_countries([second])}: "
            f"{name} Government Bond Spread"
        ),
        ylabel="Percentage points",
        xlabel=None,
        width=1,
        annotate=True,
        rounding=2,
        legend=False,
        y0=True,
        rheader=note,
        lfooter=data_to_footer(pair),
        rfooter=source_footer(pair),
        show=SHOW,
    )


def plot_all_yields(all_data: dict[str, pd.DataFrame]) -> None:
    """Chart every fetched tenor."""
    for name, data in all_data.items():
        plot_yields(name, data, TENOR_NOTES[name])


plot_all_yields(yields)
plot_yields(CHINA_US_TENOR, china_us, CHINA_US_NOTE)
plot_spread(SPREAD_TENOR, yields[SPREAD_TENOR], SPREAD_LEGS, SPREAD_NOTE)

## Risk-neutral forwards

The charts above plot yields as the market quotes them. A quoted yield is two
things at once - what the market expects the short rate to average over the
term, and the premium it demands for holding duration - so two countries' 10-year
yields can differ because their expectations differ or because their premia do,
and the yield alone cannot say which.

Both the AOFM and the New York Fed publish a decomposition that separates them.
The AOFM fits an affine term-structure model to the nominal Australian Treasury
Bond curve and publishes, daily from July 1992 and at every tenor from 1 to 10
years, a fitted zero-coupon yield (`FY`), a term premium (`TP`) and a
risk-neutral yield (`RNY`). The New York Fed's ACM model does the same for US
Treasuries, daily. From the risk-neutral curve comes the **5y5y forward** - the
average expected short rate over the five years *beginning five years from now*:

$$\text{5y5y} = 2 \times \text{RNY}_{10} - \text{RNY}_{5}$$

Five years out is far enough that the current cycle should have washed through,
so what is left is close to the market's read on where the policy rate settles.

**Two charts, both monthly.** The first sets the Australian forward against the
two standing reference points a neutral rate gets judged against: the cash rate
it is a comparison for, and trend real GDP per capita growth plus the inflation
target, which is the golden-rule statement of where a nominal neutral rate should
sit. The second puts the Australian and US forwards side by side - the same
object, computed the same way, which two 10-year yields are not.

**Caveats**

- **Neither line is r\*.** The forward is a price. It still carries whatever the
  market believes about the cyclical position over years five to ten, and
  whatever premium the model failed to strip. The growth line is an accounting
  identity about where neutral *should* sit, not a measurement of where it is.
  They disagree with each other as readily as published r\* models do.
- **The series is revised in full, every month.** The AOFM's own note says so:
  the decomposition is estimated by regression, so every historical value moves
  when new data enters the sample. That is fine for a level or a shape
  comparison. It is not a real-time series.
- **The methods are not identical across the two countries.** The AOFM publishes
  a bias-corrected decomposition (`bc`, the default here) alongside plain ACM
  (`ols`); the Fed publishes only plain ACM. The correlations move by less than
  0.02 on the AOFM `ols` sheet, so the mismatch is not what drives them.
- **The growth line is per capita, not aggregate**, and the distinction decides
  the answer rather than decorating it. Australia's population growth sits
  between the two and is worth more than the entire spread across published r\*
  estimates.
- **The cash rate is the announced target (RBA A2), not the realised overnight
  rate.** It is the policy rate itself, a step function that moves only when the
  Board decides. F1.1's "Interbank Overnight Cash Rate" is the tempting
  alternative and is the wrong series here, because it is a monthly *mean* of the
  realised rate. For most of this chart's history that distinction was nearly
  invisible: the Board met on the first Tuesday, so a new target applied to all
  but a day or two of the month and the mean sat almost on top of it. The 2024
  move to eight meetings a year on later dates widens the gap, and the mean now
  reports a blend of the old target and the new one in any month a decision lands
  in.
- **The three series on the first chart reach the monthly grid differently.** The
  forward is a mean of daily data; the cash rate is the target as at month end;
  the growth trend is quarterly, with each quarter's value on its own last month
  and the two months between interpolated, so those months are drawn rather than
  measured. The fill stops at the last published quarter rather than repeating
  its value forward, which is why the orange line ends a month or two short of
  the other two.
- **Every forward is a monthly mean of daily values**, on both charts and for
  both countries, and the final month is dropped while it is still running. A
  mean is the more meaningful monthly reading - it does not hand the month to
  whatever idiosyncratic move happened to land on the last trading day - and
  using it on both legs is what keeps the second chart like-for-like. The US
  side is read from the ACM workbook's daily sheet for that reason; the monthly
  sheet, which the workbook lists first, would force a month's mean to be
  compared with a single end-month observation, and over this sample that
  mismatch alone costs about 0.10 of the monthly-change correlation.

In [11]:
def _aofm_reachable(url: str) -> bool:
    """Whether the AOFM workbook can be fetched, or served from cache, at `url`."""
    try:
        get_file(url, cache_dir=CACHE_DIR, cache_prefix=AOFM_CACHE_PREFIX)
    except (OSError, ValueError) as error:
        print(f"AOFM workbook not available at {url} ({type(error).__name__}: {error})")
        return False
    return True


@cache
def aofm_url() -> str:
    """A working URL for the AOFM workbook, re-found on the data hub if stale.

    The AOFM serves the file from a path containing the node's creation date, so
    a site rebuild moves it. Rather than fail, the link is looked up on the data
    hub page the way a reader would. The constant is tried first, so the normal
    path costs one request rather than two.
    """
    if _aofm_reachable(AOFM_URL):
        return AOFM_URL

    page = get_file(AOFM_HUB, cache_dir=CACHE_DIR, cache_prefix=AOFM_CACHE_PREFIX)
    hrefs = re.findall(
        r'href="([^"]*term[%20_ ]*premium[^"]*\.xlsx)"',
        page.decode("utf-8", errors="replace"),
        flags=re.IGNORECASE,
    )
    if not hrefs:
        raise ValueError(
            f"The AOFM term premium workbook is not at {AOFM_URL}, and no "
            f"replacement link was found on {AOFM_HUB}. Check the data hub by hand."
        )
    href = hrefs[0]
    return href if href.startswith("http") else f"https://www.aofm.gov.au{href}"


@cache
def aofm_frame(method: str) -> pd.DataFrame:
    """One AOFM decomposition sheet, on a DatetimeIndex and numeric throughout."""
    if method not in AOFM_METHOD_SHEETS:
        raise ValueError(
            f"Unknown AOFM method {method!r}; expected one of "
            f"{', '.join(sorted(AOFM_METHOD_SHEETS))}"
        )
    # Row 0 is the method's title banner, so the column names sit on row 1.
    raw = pd.read_excel(
        io.BytesIO(
            get_file(aofm_url(), cache_dir=CACHE_DIR, cache_prefix=AOFM_CACHE_PREFIX)
        ),
        sheet_name=AOFM_METHOD_SHEETS[method],
        header=1,
    )
    frame = raw.rename(columns={raw.columns[0]: "DATE"})
    dates = pd.to_datetime(frame["DATE"], errors="coerce")
    frame = frame.loc[dates.notna()].copy()
    frame.index = pd.DatetimeIndex(dates.loc[dates.notna()])
    frame = frame.drop(columns="DATE").apply(pd.to_numeric, errors="coerce")
    if frame.empty:
        raise ValueError(
            f"The AOFM {AOFM_METHOD_SHEETS[method]} sheet holds no dated rows"
        )
    return frame


def five_year_five_year(five: pd.Series, ten: pd.Series) -> pd.Series:
    """The 5y5y forward implied by 5- and 10-year zero-coupon risk-neutral yields.

    A ten-year average is the five-year average and the 5y5y forward in equal
    parts, so the standard forward identity is 5y5y = 2 * RNY10 - RNY5. The same
    arithmetic serves both countries, which is the point: it is what makes the
    Australian and US series the same object rather than two similar ones.
    """
    forward = (2.0 * ten - five).dropna()
    if forward.empty:
        raise ValueError("The 5- and 10-year risk-neutral yields share no dates")
    return forward


def aofm_5y5y_forward(method: str = AOFM_METHOD) -> pd.Series:
    """The AOFM 5y5y risk-neutral forward, daily, per cent per year."""
    frame = aofm_frame(method)
    return five_year_five_year(frame["RNY5"], frame["RNY10"]).rename(FORWARD_LABEL)


@cache
def acm_frame() -> pd.DataFrame:
    """The New York Fed ACM daily sheet, on a DatetimeIndex and numeric throughout.

    THE SHEET IS NAMED, not left to the default. The workbook carries "ACM
    Monthly" first and "ACM Daily" second, both back to 1961, so reading it
    without `sheet_name` takes the monthly one by position rather than by
    choice. The daily sheet is the one wanted: it lets the US leg be averaged
    over the month exactly as the Australian leg is, instead of a month's mean
    being set against a single end-month observation.
    """
    frame = pd.read_excel(
        io.BytesIO(
            get_file(ACM_URL, cache_dir=CACHE_DIR, cache_prefix=ACM_CACHE_PREFIX)
        ),
        sheet_name=ACM_DAILY_SHEET,
    )
    # Dates arrive as "30-Jun-1961" strings; anything unparseable is a footer row.
    dates = pd.to_datetime(frame["DATE"], format="%d-%b-%Y", errors="coerce")
    frame = frame.loc[dates.notna()].copy()
    frame.index = pd.DatetimeIndex(dates.loc[dates.notna()])
    frame = frame.drop(columns="DATE").apply(pd.to_numeric, errors="coerce")
    if frame.empty:
        raise ValueError(f"The NY Fed {ACM_DAILY_SHEET} sheet holds no dated rows")
    return frame


def to_complete_periods(daily: pd.Series, freq: str) -> pd.Series:
    """Period means of a daily series, with an incomplete final period dropped.

    Every daily series on the two forward charts is sampled this way. A month's
    mean is the more meaningful monthly reading, because it does not hand the
    month to whatever idiosyncratic move happened to land on its last trading
    day, and using it on both countries is what keeps the two forwards
    comparable - both legs averaged over the same month, rather than one
    averaged and one read off a single day.

    Both series run into the current month, so the final period is otherwise a
    mean over however many trading days have happened so far. Plotting it puts a
    point on the chart that moves for reasons which have nothing to do with the
    market changing its mind, beside a cash rate that is complete.
    """
    clean = daily.dropna()
    grouped = clean.groupby(pd.PeriodIndex(clean.index, freq=freq))
    counts, means = grouped.count(), grouped.mean()
    if len(counts) and counts.iloc[-1] < COMPLETE_PERIOD_SHARE * counts.median():
        means = means.iloc[:-1]
    return means


def acm_us_5y5y_forward() -> pd.Series:
    """The US ACM 5y5y risk-neutral forward, monthly means, per cent per year."""
    frame = acm_frame()
    forward = five_year_five_year(frame["ACMRNY05"], frame["ACMRNY10"])
    return to_complete_periods(forward, "M").rename(US_FORWARD_LABEL)


@cache
def cash_rate_monthly() -> pd.Series:
    """The RBA's announced Cash Rate Target, monthly, per cent.

    THE POLICY RATE ITSELF, not the realised overnight rate. `read_rba_ocr` reads
    table A2 (series ARBAMPCNCRT) and keeps the last announced target in each
    month, so this is a step function that moves only when the Board decides -
    which is what belongs behind a chart about where the policy rate settles.

    F1.1's "Interbank Overnight Cash Rate" is the tempting alternative and is the
    wrong series here: it is a monthly MEAN of the realised rate, so in any month
    a decision lands in it reports a blend of the old target and the new one. The
    two differ by more than 0.05pp in 66 of the 404 months from 1993, by as much
    as 0.79pp, and the mean version visibly rounds off the corners of every step.
    """
    rate = ra.read_rba_ocr(monthly=True).astype(float)
    if rate.empty:
        raise ValueError(f"RBA {RBA_CASH_TABLE} returned no cash rate target values")
    return rate.rename("Cash rate")


@cache
def gdp_per_capita() -> pd.Series:
    """Real GDP per capita: quarterly, seasonally adjusted, chain volume measures."""
    data, meta = ra.read_abs_cat(
        ABS_NA_CAT, single_excel_only=ABS_NA_TABLE, verbose=False
    )
    _, series_id, _ = ra.find_abs_id(
        meta,
        {
            ABS_NA_TABLE: mc.table,
            GDP_PER_CAPITA_DID: mc.did,
            "Seasonally Adjusted": mc.stype,
        },
    )
    series = data[ABS_NA_TABLE][series_id].dropna()
    if series.empty:
        raise ValueError(f"ABS {ABS_NA_CAT} returned no {GDP_PER_CAPITA_DID!r} values")
    return series


def trend_growth_nominal(index: pd.PeriodIndex) -> pd.Series:
    """Trend real per-capita GDP growth plus the inflation target, on a monthly `index`.

    The second reference point for a nominal neutral rate, and the only line on
    these charts that comes from the economy rather than from an asset price.

    PER CAPITA, NOT AGGREGATE, and the distinction decides the answer rather than
    decorating it. The consumption-Euler link that ties a neutral rate to growth
    is about growth per head; the version quoted in passing is aggregate.
    Australia's population growth sits between them, and is worth more than the
    entire spread across published r* estimates.

    QUARTERLY SOURCE ON A MONTHLY GRID. The national accounts are quarterly, so
    each quarter's value is pinned to its own last month and the two months
    between are filled by straight-line interpolation. No published value moves,
    and `limit_area="inside"` stops the fill running past the last published
    quarter - without it the line is extended with a flat repeat of its own last
    value, which reads as news about months the ABS has not measured.
    """
    per_capita = gdp_per_capita().astype(float)
    per_capita.index = pd.PeriodIndex(per_capita.index, freq="Q")
    yearly = (per_capita / per_capita.shift(4) - 1) * 100
    trend = (yearly.rolling(TREND_WINDOW_QTRS).mean() + INFLATION_TARGET).dropna()
    trend.index = trend.index.asfreq("M", how="end")
    monthly = trend.reindex(trend.index.union(index)).interpolate(
        "linear", limit_area="inside"
    )
    return monthly.reindex(index).rename(TREND_G_LABEL)

In [12]:
def plot_rstar_proxies() -> None:
    """Chart the two macroeconomic proxies for a nominal neutral rate.

    Neither line is an estimate of r*: the forward is a price, close to the
    expected average policy rate five to ten years out, and trend growth plus the
    target is the golden-rule statement of where a nominal neutral rate should
    sit. They are the two standing reference points a neutral rate gets judged
    against, and they disagree with each other as readily as published models do.
    The cash rate sits behind them because it is what both are a comparison for.

    MONTHLY, and the three series reach that grid differently: the forward is a
    mean of daily data, the cash rate is the announced target as at each month
    end, and the growth trend is quarterly and interpolated between quarters. The
    footer says so, because only the first two are observations of the month.

    THE CASH RATE IS DRAWN AS STEPS, the other two as lines. A target holds flat
    until the Board changes it, so joining consecutive months with a slope draws
    a gradual move that never happened. `steps-post` holds each month's value
    across to the next, which is what the policy rate did. The other two series
    are genuinely continuous and stay on the default drawstyle.

    This one chart sets its own colours, because the cash rate has to read as
    background: grey and dashed, behind the two proxies rather than beside them.

    LAYERED RATHER THAN FINALISED IN ONE STEP, because of the boxed note in the
    bottom-left corner: `finalise_plot` takes no text argument and mgplot has no
    annotation function, so the note is the one raw-matplotlib call here. The
    note earns its place - the growth line holds the inflation term at the
    target from 1993, and expectations did not settle there until the second
    half of the 1990s, so without it the early gap between the two proxies
    reads as disagreement rather than as the constant we chose.
    """
    forward = to_complete_periods(aofm_5y5y_forward(), "M").rename(FORWARD_LABEL)
    index = pd.PeriodIndex(forward.index, freq="M")
    frame = pd.DataFrame(
        {
            FORWARD_LABEL: forward,
            "Cash rate": cash_rate_monthly().reindex(index),
            TREND_G_LABEL: trend_growth_nominal(index),
        }
    ).loc[pd.Period(RSTAR_START, freq="M"):]

    spread = (frame[FORWARD_LABEL] - frame["Cash rate"]).dropna()
    axes = mg.line_plot(
        frame,
        color=["darkblue", "darkgrey", "darkorange"],
        style=["-", "--", "-."],
        width=[2.0, 1.5, 1.8],
        drawstyle=["default", "steps-post", "default"],
        annotate=True,
        rounding=2,
    )
    axes.text(
        *RSTAR_NOTE_XY,
        RSTAR_NOTE,
        transform=axes.transAxes,
        fontsize=RSTAR_NOTE_FONTSIZE,
        ha="left",
        va="bottom",
        bbox={
            "boxstyle": "round,pad=0.4",
            "facecolor": "white",
            "edgecolor": "darkgrey",
            "alpha": 0.8,
        },
    )
    mg.finalise_plot(
        axes,
        title="Macroeconomic Proxies for Nominal r*",
        ylabel="Per cent, nominal",
        xlabel=None,
        legend={"loc": "best", "fontsize": "small"},
        lheader=(
            f"Forward less cash rate: latest {spread.iloc[-1]:+.2f}, "
            f"mean {spread.mean():+.2f}pp"
        ),
        lfooter=(
            "Australia. Monthly. Forward is a monthly average; cash rate is "
            "the announced target; growth trend is quarterly, interpolated. "
        ),
        rfooter=f"AOFM ({AOFM_METHOD.upper()}); RBA {RBA_CASH_TABLE}; ABS {ABS_NA_CAT}",
        show=SHOW,
    )


def plot_forward_correlation() -> None:
    """The Australian and US 5y5y risk-neutral forwards, monthly, side by side.

    The same object computed the same way in both countries - the average
    expected short rate over years five to ten, with the term premium removed by
    an affine model - so the two lines are comparable in a way that two 10-year
    yields are not.

    BOTH LEGS ARE MONTHLY MEANS OF DAILY DATA, which is what makes the
    comparison a like-for-like one. Sampling the two countries differently is
    not free: over this sample, averaging one leg and taking the other's last
    trading day costs about 0.10 of the monthly-change correlation, and the
    lost tenth is sampling mismatch rather than anything the two markets did.

    BOTH CORRELATIONS ARE REPORTED, because they answer different questions. The
    level correlation is inflated by two series that both drift down across
    thirty years; the correlation of monthly changes asks whether the two markets
    move together, which is the harder and more interesting test. Neither is high
    over the full sample: the levels track each other closely only from about
    2015, and the two forwards are plainly apart from 2002 to 2019.
    """
    australia = to_complete_periods(aofm_5y5y_forward(), "M").rename(FORWARD_LABEL)
    united_states = acm_us_5y5y_forward()
    frame = pd.concat([australia, united_states], axis=1).dropna()
    frame = frame.loc[pd.Period(CORRELATION_START, freq="M"):]

    levels = frame[FORWARD_LABEL].corr(frame[US_FORWARD_LABEL])
    changes = frame.diff().dropna()
    monthly = changes[FORWARD_LABEL].corr(changes[US_FORWARD_LABEL])
    mg.line_plot_finalise(
        frame,
        title="Australian and US 5y5y Risk-Neutral Forwards",
        ylabel="Per cent, nominal",
        xlabel=None,
        width=1,
        annotate=True,
        rounding=2,
        legend={"loc": "best", "fontsize": "small"},
        lheader=(
            f"Correlation over {frame.index[0]} to {frame.index[-1]}: "
            f"levels {levels:+.2f}, monthly changes {monthly:+.2f}"
        ),
        lfooter=(
            "Monthly averages. Both are 2 x RNY10 - RNY5. "
            f"AU bias-corrected ({AOFM_METHOD.upper()}), US plain ACM. "
        ),
        rfooter="AOFM; NY Fed ACM",
        show=SHOW,
    )


plot_rstar_proxies()
plot_forward_correlation()

## Watermark

In [13]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-09-22 07:06:20

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.17.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 27.0.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.33
pandas  : 3.0.5
pathlib : 1.0.1
re      : 2.2.1
readabs : 0.2.6
requests: 2.34.2
typing  : 3.10.0.0
yfinance: 1.7.0

Watermark: 2.6.0

